# Concurrency Tests

This notebook compares **pessimistic** and **optimistic** concurrency control for the `library_sim` database.

The test uses multiple Python threads to simulate concurrent database activity and reports metrics such as runtime, successful operations, errors, deadlocks, timeouts, conflicts, and logged transactions.

## 1. Imports

In [1]:
import mysql.connector
import threading
import time
from getpass import getpass

## 2. MySQL Connection Settings

The notebook prompts for the MySQL password instead of reading from a `credentials.ini` file.

In [2]:
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": getpass("MySQL password: "),
    "database": "library_sim"
}

NUM_THREADS = 20
OPERATIONS_PER_THREAD = 5

## 3. Metrics

These dictionaries track results for both concurrency strategies.

In [3]:
metrics = {
    "optimistic": {
        "success": 0,
        "errors": 0,
        "deadlocks": 0,
        "timeouts": 0
    },
    "pessimistic": {
        "success": 0,
        "errors": 0,
        "deadlocks": 0,
        "timeouts": 0
    }
}

metric_lock = threading.Lock()

## 4. Database Connection Helper

In [4]:
def get_connection():
    return mysql.connector.connect(**DB_CONFIG)

## 5. Reset Test Environment

Before testing, clear the log tables and reset the selected book copies.

In [5]:
def reset_test_environment():
    try:
        conn = get_connection()
        cursor = conn.cursor()

        cursor.execute("DELETE FROM transaction_log")
        cursor.execute("DELETE FROM conflict_log")
        cursor.execute("DELETE FROM deadlock_log")

        cursor.execute("""
            UPDATE book_copy
            SET version = 0,
                status = 'Available'
            WHERE copy_id IN (1,2)
        """)

        conn.commit()

    except Exception as e:
        print(f"Reset error: {e}")
        try:
            conn.rollback()
        except:
            pass

    finally:
        try:
            cursor.close()
            conn.close()
        except:
            pass

## 6. Pessimistic Worker

Each thread repeatedly calls the `checkout_pessimistic` stored procedure.

In [6]:
def pessimistic_worker(thread_id):
    conn = get_connection()
    cursor = conn.cursor()

    for i in range(OPERATIONS_PER_THREAD):
        try:
            cursor.callproc("checkout_pessimistic", [1])

            with metric_lock:
                metrics["pessimistic"]["success"] += 1

        except Exception as e:
            error_text = str(e)

            with metric_lock:
                metrics["pessimistic"]["errors"] += 1

                if "1205" in error_text:
                    metrics["pessimistic"]["timeouts"] += 1

                if "1213" in error_text:
                    metrics["pessimistic"]["deadlocks"] += 1

            conn.rollback()

    cursor.close()
    conn.close()

## 7. Optimistic Worker

Each thread repeatedly calls the `checkout_optimistic` stored procedure.

In [7]:
def optimistic_worker(thread_id):
    conn = get_connection()
    cursor = conn.cursor()

    for i in range(OPERATIONS_PER_THREAD):
        try:
            cursor.callproc(
                "checkout_optimistic",
                [1, thread_id, "READ COMMITTED"]
            )

            for result in cursor.stored_results():
                result.fetchall()

            with metric_lock:
                metrics["optimistic"]["success"] += 1

            conn.commit()

        except Exception as e:
            print(f"Optimistic Error: {e}")

            with metric_lock:
                metrics["optimistic"]["errors"] += 1

    cursor.close()
    conn.close()

## 8. Run Each Concurrency Test

In [8]:
def run_pessimistic():
    threads = []
    start_time = time.time()

    for i in range(NUM_THREADS):
        t = threading.Thread(
            target=pessimistic_worker,
            args=(i,)
        )
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    return round(time.time() - start_time, 2)


def run_optimistic():
    threads = []
    start_time = time.time()

    for i in range(NUM_THREADS):
        t = threading.Thread(
            target=optimistic_worker,
            args=(i,)
        )
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    return round(time.time() - start_time, 2)

## 9. Read MySQL Log Counts

In [9]:
def get_db_metrics():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("SELECT COUNT(*) FROM conflict_log")
    conflict_count = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM deadlock_log")
    deadlock_count = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM transaction_log")
    transaction_count = cursor.fetchone()[0]

    cursor.close()
    conn.close()

    return conflict_count, deadlock_count, transaction_count

## 10. Run the Full Test

This resets the environment, runs both strategies, and prints the comparison.

In [10]:
print("\nResetting environment...\n")
reset_test_environment()

print("Running pessimistic test...\n")
pess_time = run_pessimistic()

print("Running optimistic test...\n")
opt_time = run_optimistic()

conflict_count, deadlock_count, transaction_count = get_db_metrics()

print("\n")
print("=" * 85)
print("CONCURRENCY TEST RESULTS")
print("=" * 85)

print(
    f"{'Metric':<25}"
    f"{'Pessimistic':<20}"
    f"{'Optimistic':<20}"
)

print("-" * 85)

print(f"{'Runtime (seconds)':<25}{pess_time:<20}{opt_time:<20}")
print(f"{'Success Count':<25}{metrics['pessimistic']['success']:<20}{metrics['optimistic']['success']:<20}")
print(f"{'Exception Count':<25}{metrics['pessimistic']['errors']:<20}{metrics['optimistic']['errors']:<20}")
print(f"{'Deadlocks (1213)':<25}{metrics['pessimistic']['deadlocks']:<20}{metrics['optimistic']['deadlocks']:<20}")
print(f"{'Timeouts (1205)':<25}{metrics['pessimistic']['timeouts']:<20}{metrics['optimistic']['timeouts']:<20}")

print("-" * 85)

print(f"{'Conflict Log Rows':<25}{'-':<20}{conflict_count:<20}")
print(f"{'Deadlock Log Rows':<25}{deadlock_count:<20}{'-':<20}")
print(f"{'Transaction Log Rows':<25}{transaction_count:<20}{transaction_count:<20}")

print("=" * 85)


Resetting environment...

Running pessimistic test...

Running optimistic test...



CONCURRENCY TEST RESULTS
Metric                   Pessimistic         Optimistic          
-------------------------------------------------------------------------------------
Runtime (seconds)        0.04                0.03                
Success Count            100                 100                 
Exception Count          0                   0                   
Deadlocks (1213)         0                   0                   
Timeouts (1205)          0                   0                   
-------------------------------------------------------------------------------------
Conflict Log Rows        -                   93                  
Deadlock Log Rows        0                   -                   
Transaction Log Rows     200                 200                 


/var/folders/qy/yl4hyv4s5kz52wg1qlh36y_w0000gn/T/ipykernel_43336/486957982.py:12: DeprecationWarning: Call to deprecated function stored_results. Reason: The property counterpart 'stored_results' will be added in a future release, and this method will be removed.
  for result in cursor.stored_results():
